# TA-03: Data-Driven Monte Carlo Simulation
## Estimasi Risiko Klaim Asuransi (Medical Cost Personal Dataset)

**Pertanyaan bisnis:** Jika ada 100 pasien baru, berapa probabilitas **total klaim melebihi anggaran perusahaan**?

Alur: Load data Kaggle -> Cleaning -> Ekstraksi parameter (mean, std) -> Simulasi Monte Carlo -> Analisis risiko.

Dataset: https://www.kaggle.com/datasets/mirichoi0218/insurance (file `insurance.csv`)

## 0. Import Library & Pengaturan Seed

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ganti dengan NIM kamu supaya hasil konsisten (reproducible)
NIM = 301240055
np.random.seed(NIM)

# Parameter simulasi
N_PATIENTS = 100        # jumlah pasien baru
N_SIM = 10_000          # jumlah skenario (minimal 5.000)
BUDGET_FACTOR = 1.05    # anggaran = 105% dari ekspektasi total klaim (asumsi, boleh diubah)

## 1. Eksplorasi Data Kaggle

### 1.1 Load Data
Unduh `insurance.csv` dari Kaggle, lalu upload ke Colab (ikon folder di sidebar kiri) atau letakkan satu folder dengan notebook ini.

In [ ]:
df = pd.read_csv("insurance.csv")
print("Ukuran data:", df.shape)
df.head()

In [ ]:
df.info()
df.describe()

### 1.2 Data Cleaning
Cek *null*, duplikat, lalu buang **outlier ekstrem** pada kolom `charges` dengan aturan IQR (batas 3 x IQR, bukan 1,5 x, supaya yang dibuang hanya yang benar-benar ekstrem).

In [ ]:
print("Missing value per kolom:")
print(df.isnull().sum())
print("\nJumlah baris duplikat:", df.duplicated().sum())

df = df.dropna().drop_duplicates().reset_index(drop=True)

# Outlier ekstrem (3 x IQR)
Q1, Q3 = df["charges"].quantile([0.25, 0.75])
IQR = Q3 - Q1
batas_atas = Q3 + 3 * IQR
batas_bawah = Q1 - 3 * IQR
print(f"\nBatas outlier ekstrem charges: {batas_bawah:,.2f} s/d {batas_atas:,.2f}")

df_clean = df[(df["charges"] >= batas_bawah) & (df["charges"] <= batas_atas)].copy()
print(f"Baris sebelum cleaning: {len(df)}, sesudah: {len(df_clean)} (dibuang {len(df) - len(df_clean)})")

### 1.3 Parameter Extraction (Mean & Std Dev)

In [ ]:
charges = df_clean["charges"].values
mu = charges.mean()
sigma = charges.std(ddof=1)
skewness = pd.Series(charges).skew()

print(f"Mean (mu)            : {mu:,.2f}")
print(f"Std Dev (sigma)      : {sigma:,.2f}")
print(f"Skewness             : {skewness:.3f}")
print(f"Median               : {np.median(charges):,.2f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].hist(charges, bins=40, color="steelblue", edgecolor="white", density=True)
ax[0].axvline(mu, color="red", linestyle="--", label=f"Mean = {mu:,.0f}")
ax[0].set_title("Histogram charges (data asli)")
ax[0].set_xlabel("charges")
ax[0].legend()

ax[1].hist(np.log(charges), bins=40, color="darkorange", edgecolor="white", density=True)
ax[1].set_title("Histogram log(charges)")
ax[1].set_xlabel("log(charges)")

plt.tight_layout()
plt.show()

**Interpretasi distribusi (isi dengan hasil kamu):** Histogram `charges` biasanya **menceng ke kanan (right-skewed)**, yaitu banyak klaim kecil dan sedikit klaim sangat besar (umumnya pasien perokok). Artinya distribusi Normal kurang pas untuk satu pasien. Karena itu, selain Normal (sesuai ketentuan tugas), simulasi ini juga membandingkan **Lognormal** dan **Bootstrap** sebagai pembanding.

## 2. Simulasi Monte Carlo

Setiap skenario = 1 "periode" dengan 100 pasien baru. Untuk tiap pasien dibangkitkan `charges` acak, lalu dijumlahkan menjadi **total klaim** skenario tersebut. Diulang `N_SIM` kali.

**Anggaran perusahaan** didefinisikan sebagai `BUDGET_FACTOR x N_PATIENTS x mu`. Dataset tidak memberi angka anggaran, jadi ini adalah asumsi yang harus dijelaskan di laporan (di bagian 3 ada analisis sensitivitas).

In [ ]:
BUDGET = BUDGET_FACTOR * N_PATIENTS * mu
print(f"Anggaran perusahaan (asumsi): {BUDGET:,.2f}")
print(f"Ekspektasi total klaim       : {N_PATIENTS * mu:,.2f}")

### 2.1 Metode A: Distribusi Normal (sesuai ketentuan tugas)

In [ ]:
# Matriks (N_SIM x N_PATIENTS); nilai negatif dipotong ke 0 karena klaim tidak boleh negatif
sim_normal = np.random.normal(mu, sigma, size=(N_SIM, N_PATIENTS))
sim_normal = np.clip(sim_normal, 0, None)
total_normal = sim_normal.sum(axis=1)

print(f"Rata-rata total klaim (Normal): {total_normal.mean():,.2f}")

**Catatan penting:** Pada data asli, std (sekitar 12 ribu) hampir sama besar dengan mean (sekitar 13 ribu), sehingga banyak nilai Normal yang negatif. Setelah dipotong ke 0, rata-rata klaim simulasi **naik di atas mean data asli**, dan probabilitas risiko jadi terlalu tinggi (bias ke atas). Bandingkan `Rata-rata total klaim` ketiga metode di bagian 3: Bootstrap paling dekat dengan ekspektasi sebenarnya. Jelaskan temuan ini di laporan sebagai alasan memakai metode pembanding.

### 2.2 Metode B: Distribusi Lognormal (lebih cocok untuk data menceng)

In [ ]:
log_mu = np.log(charges).mean()
log_sigma = np.log(charges).std(ddof=1)
print(f"Parameter log: mu = {log_mu:.4f}, sigma = {log_sigma:.4f}")

sim_lognormal = np.random.lognormal(log_mu, log_sigma, size=(N_SIM, N_PATIENTS))
total_lognormal = sim_lognormal.sum(axis=1)

print(f"Rata-rata total klaim (Lognormal): {total_lognormal.mean():,.2f}")

### 2.3 Metode C: Bootstrap (ambil acak dari data asli, tanpa asumsi distribusi)

In [ ]:
idx = np.random.randint(0, len(charges), size=(N_SIM, N_PATIENTS))
total_bootstrap = charges[idx].sum(axis=1)

print(f"Rata-rata total klaim (Bootstrap): {total_bootstrap.mean():,.2f}")

## 3. Analisis Risiko & Kesimpulan

Probabilitas kejadian:

$$P(\text{Total Klaim} > \text{Anggaran}) = \frac{\text{Jumlah skenario dengan total klaim} > \text{anggaran}}{\text{Total skenario}}$$

In [ ]:
hasil = {
    "Normal": total_normal,
    "Lognormal": total_lognormal,
    "Bootstrap": total_bootstrap,
}

rows = []
for nama, total in hasil.items():
    p = np.mean(total > BUDGET)
    se = np.sqrt(p * (1 - p) / N_SIM)   # galat baku estimasi probabilitas
    rows.append({
        "Metode": nama,
        "Rata-rata total klaim": total.mean(),
        "Std total klaim": total.std(),
        "Persentil 95": np.percentile(total, 95),
        "P(klaim > anggaran)": p,
        "Std error P": se,
    })

tabel = pd.DataFrame(rows).set_index("Metode")
tabel.style.format({
    "Rata-rata total klaim": "{:,.0f}", "Std total klaim": "{:,.0f}",
    "Persentil 95": "{:,.0f}", "P(klaim > anggaran)": "{:.2%}", "Std error P": "{:.3%}",
})

### 3.1 Visualisasi Hasil dengan Garis Batas (Threshold)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
warna = {"Normal": "steelblue", "Lognormal": "seagreen", "Bootstrap": "darkorange"}

for ax, (nama, total) in zip(axes, hasil.items()):
    ax.hist(total, bins=50, color=warna[nama], edgecolor="white", alpha=0.85)
    ax.axvline(BUDGET, color="red", linestyle="--", linewidth=2, label=f"Anggaran = {BUDGET:,.0f}")
    ax.axvspan(BUDGET, total.max(), color="red", alpha=0.12, label="Area risiko")
    p = np.mean(total > BUDGET)
    ax.set_title(f"{nama}  |  P(risiko) = {p:.2%}")
    ax.set_xlabel("Total klaim 100 pasien")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Frekuensi skenario")
plt.tight_layout()
plt.show()

### 3.2 Analisis Sensitivitas terhadap Anggaran
Karena anggaran adalah asumsi, lihat bagaimana probabilitas berubah jika anggaran diubah.

In [ ]:
faktor = [1.00, 1.025, 1.05, 1.075, 1.10, 1.15, 1.20]
sens = pd.DataFrame({
    "Faktor anggaran": faktor,
    "Anggaran": [f * N_PATIENTS * mu for f in faktor],
})
for nama, total in hasil.items():
    sens[f"P ({nama})"] = [np.mean(total > f * N_PATIENTS * mu) for f in faktor]

sens.style.format({"Faktor anggaran": "{:.3f}", "Anggaran": "{:,.0f}",
                   **{c: "{:.2%}" for c in sens.columns if c.startswith("P (")}})

In [ ]:
plt.figure(figsize=(7, 4))
for nama in hasil:
    plt.plot(sens["Faktor anggaran"], sens[f"P ({nama})"], marker="o", label=nama)
plt.axvline(BUDGET_FACTOR, color="gray", linestyle=":", label="Skenario utama")
plt.xlabel("Faktor anggaran (x ekspektasi total klaim)")
plt.ylabel("P(total klaim > anggaran)")
plt.title("Sensitivitas risiko terhadap anggaran")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 3.3 Pengaruh Outlier (Opsional, untuk Pembahasan)
Outlier ekstrem dibuang di langkah cleaning. Cek apakah itu membuat risiko terlihat lebih kecil dari seharusnya.

In [ ]:
charges_all = df["charges"].values   # sebelum buang outlier ekstrem
idx_all = np.random.randint(0, len(charges_all), size=(N_SIM, N_PATIENTS))
total_all = charges_all[idx_all].sum(axis=1)
budget_all = BUDGET_FACTOR * N_PATIENTS * charges_all.mean()
print(f"P(risiko) bootstrap, data tanpa buang outlier : {np.mean(total_all > budget_all):.2%}")
print(f"P(risiko) bootstrap, data setelah cleaning    : {np.mean(total_bootstrap > BUDGET):.2%}")

## 4. Kesimpulan (edit sesuai hasil kamu)

1. Parameter `charges` hasil ekstraksi: mean = ..., std = ..., skewness = ... (distribusi menceng ke kanan).
2. Dengan 100 pasien baru dan anggaran sebesar 105% dari ekspektasi klaim, probabilitas total klaim melebihi anggaran adalah sekitar ...% (Normal), ...% (Lognormal), ...% (Bootstrap).
3. Hasil ketiga metode (sama / berbeda) karena ... (petunjuk: total 100 pasien mendekati Normal berkat Teorema Limit Pusat, sehingga perbedaan antarmetode biasanya kecil).
4. Risiko sangat sensitif terhadap besar anggaran: menaikkan anggaran dari 100% ke 120% menurunkan probabilitas dari ...% menjadi ...%.
5. Rekomendasi: perusahaan sebaiknya menetapkan anggaran minimal ... agar probabilitas risiko di bawah 5%.